In [0]:
%sql

USE CATALOG rocket;

CREATE SCHEMA IF NOT EXISTS silver;

CREATE TABLE IF NOT EXISTS silver.fat_avaliacoes;
CREATE TABLE IF NOT EXISTS silver.fat_pedidos;
CREATE TABLE IF NOT EXISTS silver.dim_clientes;
CREATE TABLE IF NOT EXISTS silver.dim_catalogo_produtos;
CREATE TABLE IF NOT EXISTS silver.fat_suporte_tickets;
CREATE TABLE IF NOT EXISTS silver.fat_clickstream;

In [0]:
from pyspark.sql.functions import col, lower, trim, translate, when, year, lit, split, explode, regexp_replace, regexp_extract, initcap, row_number, desc, sum, to_date, date_format, to_timestamp, coalesce, abs as spark_abs, round as spark_round
from pyspark.sql.window import Window
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Tabela Clientes

In [0]:
def normalizar(c):
    return trim(lower(
        translate(c,
            "áàâãäéèêëíìîïóòôõöúùûüç",
            "aaaaaeeeeiiiiooooouuuuc"
        )
    ))

df = spark.read.table('rocket.bronze.tb_clientes')

In [0]:
df_silver = df.dropDuplicates()

display(df_silver.count())

In [0]:
# id_cliente solucinando duplicatas usando deduplicação sênior com data_castro como referéncia

window_spec = Window.partitionBy("id_cliente").orderBy(desc("data_cadastro"))

df_silver = df_silver.withColumn("row_num", row_number().over(window_spec)) \
                     .filter(col("row_num") == 1) \
                     .drop("row_num")

display(df_silver.select("id_cliente", "data_cadastro"))

In [0]:
#nome e sobrenome usando initcap para padronizar

df_silver = df_silver.withColumn("nome", initcap(lower(col("nome")))) \
              .withColumn("sobrenome", initcap(lower(col("sobrenome"))))
display(df_silver.select("nome", "sobrenome"))

In [0]:
from pyspark.sql.functions import col, when, trim, regexp_replace, lower

# Limpezas que regex consegue corrigir

df_silver = (df_silver
    .withColumn("email", trim(col("email")))                                       
    .withColumn("email", regexp_replace(col("email"), r"\s+", ""))                  
    .withColumn("email", regexp_replace(col("email"), r"@@+", "@"))               
    .withColumn("email", regexp_replace(col("email"), r"\.{2,}", "."))              
    .withColumn("email", regexp_replace(col("email"), r"\.$", ""))                 
    .withColumn("email", lower(col("email")))                                    
)

# Validação com regex

EMAIL_REGEX = r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$"

df_silver = df_silver.withColumn(
    "email_valido",
    col("email").rlike(EMAIL_REGEX)
)

# Substituir por null quando inválido
# mas manter o registro se tem nome + telefone + endereço

tem_nome = (
    col("nome").isNotNull() & (trim(col("nome")) != "") &
    col("sobrenome").isNotNull() & (trim(col("sobrenome")) != "")
)

tem_telefone = col("telefone").isNotNull() & (trim(col("telefone")) != "")

tem_endereco = col("endereco").isNotNull() & (trim(col("endereco")) != "")

tem_contato_alternativo = tem_nome & tem_telefone & tem_endereco

df_silver = df_silver.withColumn(
    "email",
    when(~col("email_valido") & tem_contato_alternativo, None)
    .when(~col("email_valido"), None)
    .otherwise(col("email"))
)

# Descartar registros sem nenhum meio de contato útil

df_silver = df_silver.filter(
    col("email").isNotNull() | tem_contato_alternativo
)

# Remove coluna auxiliar
df_silver = df_silver.drop("email_valido")

display(df_silver)
print(df_silver.count())

In [0]:
#telefone sendo separado em duas colunas (telefone_formatado e telefone_ramal) usando regex 
df_silver = df_silver.withColumn(
    "telefone_ramal",
    coalesce(
        regexp_extract(col("telefone"), r"r\.\s*(\d+)", 1),
        lit("")
    )
)

df_silver = df_silver.withColumn(
    "telefone_formatado",
    regexp_replace(
        regexp_replace(
            regexp_replace(
                regexp_replace(col("telefone"), r"r\.\s*\d+", ""), 
                r"[^\d]", ""                                        
            ),
            r"^55(\d{10,11})$", "$1"                                
        ),
        r"(\d{2})(\d{4,5})(\d{4})",                                 
        "($1)$2-$3"
    )
)

df_silver = df_silver.drop("telefone")

display(df_silver.select("telefone_ramal", "telefone_formatado"))

In [0]:
#transformar data_nascimento de nascimento inválidas em nulas 

ANO_MIN = 1920
ANO_MAX = 2020

df_silver = df_silver.withColumn(
    "data_nascimento",
    when(
        (year(col("data_nascimento")) >= ANO_MIN) &
        (year(col("data_nascimento")) <= ANO_MAX) &
        (col("data_nascimento") <= col("data_cadastro")),
        col("data_nascimento")
    ).otherwise(lit(None))
)

display(df_silver.select("id_cliente", "nome", "sobrenome", "data_nascimento", "data_cadastro"))

In [0]:
# endereço formatado
# americano
regex_num_usa = r'^(\d+)'
regex_comp_usa = r'(?i)(apt\.?\s*\d+|suite\s*\d+)'
regex_rua_usa = r'^\d+\s+(.+?)(?:\s+(?i:apt\.?|suite)\s*\d+)?$'

# brasileiro
regex_tipo_br = r'^(Rua|Avenida|Alameda|Travessa)'
regex_rua_br = r'^(?:Rua|Avenida|Alameda|Travessa)\s+([A-Za-zÀ-ÿ\s]+),'
regex_num_br = r',\s*(\d+)'


df_silver = df_silver.withColumn(
    "tipo_padrao",
    when(col("endereco").rlike(r'^\d+'), "usa")
    .when(col("endereco").rlike(r'^(Rua|Avenida|Alameda|Travessa)'), "br")
    .otherwise("desconhecido")
)

df_silver = df_silver.withColumn(
    "numero",
    when(col("tipo_padrao") == "usa",
         regexp_extract(col("endereco"), regex_num_usa, 1)
    ).when(col("tipo_padrao") == "br",
         regexp_extract(col("endereco"), regex_num_br, 1)
    )
).withColumn(
    "tipo_logradouro",
    when(col("tipo_padrao") == "br",
         regexp_extract(col("endereco"), regex_tipo_br, 1)
    )
).withColumn(
    "rua",
    when(col("tipo_padrao") == "usa",
         regexp_extract(col("endereco"), regex_rua_usa, 1)
    ).when(col("tipo_padrao") == "br",
         regexp_extract(col("endereco"), regex_rua_br, 1)
    )
).withColumn(
    "complemento",
    coalesce(
        when(col("tipo_padrao") == "usa",
            lower(
                regexp_replace(
                    regexp_extract(col("endereco"), regex_comp_usa, 1),
                    r'apt\.',
                    'apt'
                )
            )
        ),
        lit("")
    )
)


df_silver = df_silver.withColumn("rua", initcap(normalizar(col("rua")))) \
       .withColumn("tipo_logradouro", initcap(normalizar(col("tipo_logradouro")))) \
       .withColumn("numero", col("numero").cast("int"))

df_silver = df_silver.withColumn(
    "erro_endereco",
    when(
        col("numero").isNull() | col("rua").isNull(),
        1
    ).otherwise(0)
)

df_silver = df_silver.drop("endereco", "tipo_padrao", "tipo_logradouro", "erro_endereco")

display(df_silver.select("rua", "numero", "complemento"))

In [0]:
# estados e cidades caso, o nome esteja trocados entre si

estados = [
    "acre","alagoas","amapa","amazonas","bahia","ceara","distrito federal",
    "espirito santo","goias","maranhao","mato grosso","mato grosso do sul",
    "minas gerais","para","paraiba","parana","pernambuco","piaui",
    "rio de janeiro","rio grande do norte","rio grande do sul",
    "rondonia","roraima","santa catarina","sao paulo","sergipe","tocantins"
]


df_silver = df_silver.withColumn("estado_norm", normalizar(col("estado"))) \
       .withColumn("cidade_norm", normalizar(col("cidade")))

# condição:
# estado inválido E cidade exatamente igual a um estado válido
condicao = (
    (~col("estado_norm").isin(estados)) &
    (col("cidade_norm").isin(estados))
)

# swap
df_silver = df_silver.withColumn(
    "estado_corrigido",
    when(condicao, col("cidade"))
    .otherwise(col("estado"))
).withColumn(
    "cidade_corrigida",
    when(condicao, col("estado"))
    .otherwise(col("cidade"))
)

df_silver = df_silver.withColumn(
    "estado_corrigido",
    initcap(normalizar(col("estado_corrigido")))
)

# substituir colunas
df_silver = df_silver.drop("estado", "cidade", "estado_norm", "cidade_norm") \
       .withColumnRenamed("estado_corrigido", "estado") \
       .withColumnRenamed("cidade_corrigida", "cidade")


In [0]:
#device_ids será uma tabela separada 1:N

df_dispositivos = df_silver.select(
    col("id_cliente"),
    explode(split(col("device_ids"), ";")).alias("device_id")
).withColumn("device_id", trim(col("device_id"))) \
 .filter(col("device_id") != "")

display(df_dispositivos)

In [0]:
# origem sendo mudada pra tudo minusculo e sem acento
origens_validas = ["app", "web", "indicacao"]

df_silver = df_silver.withColumn("origem_norm", normalizar(col("origem")))

df_silver = df_silver.withColumn(
    "origem",
    when(col("origem_norm").isin(origens_validas), col("origem_norm"))
    .otherwise(None)
).drop("origem_norm")

display(df_silver.select("origem").distinct())

In [0]:
df_silver = df_silver.select(
    "id_cliente",
    "nome",
    "sobrenome",
    "email",
    "genero",
    "data_nascimento",
    "data_cadastro",
    "telefone_formatado",
    "telefone_ramal",
    "rua",
    "numero",
    "complemento",
    "cidade",
    "estado",
    "pais",
    "origem",
    "timestamp_ingestion",
)

display(df_silver)

In [0]:
(
    df_silver
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("rocket.silver.dim_clientes")
)
 
print("rocket.silver.dim_clientes atualizada com sucesso!") 

# Tabela Avaliação
## 

In [0]:
from pyspark.sql.functions import col,sum, when, lower, to_date, date_format, to_timestamp
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

**nota_produto:**
- Troca de valores literais por numericos correspondentes
- Limpeza de notas maiores do que 5 e menores do que 1
- Conversão do tipo da coluna para int

In [0]:
# Ajustes feitos em nota produto

df = spark.read.table('rocket.bronze.tb_avaliacoes')

sub = {
    'bom': '4',
    'ruim': '2',
    'péssimo': '1',
    'ótimo': '5'
}

df = df.replace(to_replace=sub, subset=['nota_produto'])

df = df.withColumn("nota_produto", col("nota_produto").cast("int"))

df = df.withColumn(
    "nota_produto",
    when(col("nota_produto") > 5, 5)
    .when(col("nota_produto") < 1, 1)
    .otherwise(col("nota_produto")))

df.write.format("delta").mode("overwrite").option('overwriteSchema', 'true').saveAsTable(f"rocket.silver.fat_avaliacoes")

**nota_nps:**
- Transformando notas acimad de 10 para 10
- Transformando notas abaixo de 1 para 1
- Mapeando as strings em notas: 'ótimo' vira nota 9, 'bom' vira nota 7, 'ruim' vira nota 5 e 'péssimo' vira nota 3
- Convertendo a coluna para inteiro

In [0]:
df = spark.read.table('rocket.silver.fat_avaliacoes')

sub = {
    'bom': '7',
    'ruim': '5',
    'péssimo': '3',
    'ótimo': '9'
}

df = df.replace(to_replace=sub, subset=['nota_nps'])

df = df.withColumn("nota_nps", col("nota_nps").cast("int"))

df = df.withColumn(
    "nota_nps",
    when(col("nota_nps") > 10, 10)
    .when(col("nota_nps") < 1, 1)
    .otherwise(col("nota_nps")))

df.write.format("delta").mode("overwrite").option('overwriteSchema', 'true').saveAsTable(f"rocket.silver.fat_avaliacoes")

**recomenda:**
- Padronização de valores para mais fácil compreensão
- Valores padronziados como sim e nao

In [0]:
df = spark.read.table('rocket.silver.fat_avaliacoes')

df = df.withColumn(
    "recomenda",
    lower(col("recomenda"))
)

map_respostas = {
    "s": "sim", "yes": "sim", "1": "sim",

    "n": "nao", "no": "nao", "0": "nao"
}

df = df.replace(to_replace=map_respostas, subset=['recomenda'])

df.write.format("delta").mode("overwrite").option('overwriteSchema', 'true').saveAsTable(f"rocket.silver.fat_avaliacoes")

**data_avaliacao:**
- Removendo os horários das datas
- Removendo linhas sem datas de avaliação
- Filtrando linhas que tem suas datas de avaliação depois de 2026-04-28
- Convertendo a coluna para o tipo data

In [0]:
df = spark.read.table('rocket.silver.fat_avaliacoes')

df = df.withColumn(
    "data_avaliacao",
    when(
        col("data_avaliacao").rlike(
            r"^\d{4}/\d{2}/\d{2} \d{2}:\d{2}:\d{2}$"
        ),
        date_format(
            to_timestamp(
                col("data_avaliacao"),
                "yyyy/dd/MM HH:mm:ss"
            ),
            "yyyy-MM-dd"
        )
    ).otherwise(col("data_avaliacao"))
)

df = df.withColumn(
    "data_avaliacao",
    when(
        col("data_avaliacao").rlike(
            r"^\d{2}/\d{2}/\d{4} \d{2}:\d{2}:\d{2}$"
        ),
        date_format(
            to_timestamp(
                col("data_avaliacao"),
                "dd/MM/yyyy HH:mm:ss"
            ),
            "yyyy-MM-dd"
        )
    ).otherwise(col("data_avaliacao"))
)

df = df.withColumn(
    "data_avaliacao",
    when(
        col("data_avaliacao").rlike(
            r"^\d{2}-\d{2}-\d{4} \d{2}:\d{2}:\d{2}$"
        ),
        date_format(
            to_timestamp(
                col("data_avaliacao"),
                "MM-dd-yyyy HH:mm:ss"
            ),
            "yyyy-MM-dd"
        )
    ).otherwise(col("data_avaliacao"))
)

df = df.withColumn(
    "data_avaliacao",
    when(
        col("data_avaliacao").rlike(
            r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$"
        ),
        date_format(
            to_timestamp(
                col("data_avaliacao"),
                "yyyy-MM-dd HH:mm:ss"
            ),
            "yyyy-MM-dd"
        )
    ).otherwise(col("data_avaliacao"))
)

df = df.filter(F.col("data_avaliacao") <= "2026-04-28")
df = df.na.drop(subset=["data_avaliacao"])

df = df.withColumn("data_avaliacao", col("data_avaliacao").cast("date"))

df.write.format("delta").mode("overwrite").option('overwriteSchema', 'true').saveAsTable(f"rocket.silver.fat_avaliacoes")

In [0]:
spark.read.table("rocket.silver.fat_avaliacoes").display() 

# Tabela Pedidos

In [0]:
bronze_pedidos = spark.read.table('rocket.bronze.tb_pedidos')

bronze_pedidos.display()

- Id-pedido duplicado (Colunas iguais e colunas diferentes) ✅

- Verificar valores nulos e alguma tratativa para eles ✅

- Verificar se as chaves estrangeiras (id_client e id_produto) referenciam instâncias reais (Se existem na outra tabela) ✅

- Verificar formatação correta de valor_pedido (Para float) ❌ (Tirar o R$ e substituir , por .)

- Verificar se tem valores negativos para valor_pedido ❌ (Uma possibilidade seria usar o valor absoluto)

- Verificar formatação correta para data pedido (YYYY-MM-DD) ❌ (Usar o método demonstrado na célula de verificação)

- Verificar também dias e meses inválidos (Mês 13 e dia 32) ✅

- Verificar datas futuras ❌ (Todas as datas estão em maio de 2026 então talvez não seja necessário tratar, por conta também da data do fim do projeto, e eu lembro de eles falando algo sobre isso)

- Verificar categorias corretamente a coluna método de pagamento ❌ (Basta colocar pra upper e fazer um dicionário de substituições)

- Verificar também categoria da coluna status ❌(Basta colocar pra upper e fazer um dicionário de substituições)

- Verificar se a quantidade não é negativa e está na formatação correta (int) ❌ (Uma possibilidade seria usar o valor absoluto, e para formatação fazer um dicionário de substituições)

OBS: O título das celulas especifica qual etapa foi implementada

In [0]:
# Verifica as duplicatas
# O código agrupa por id e verifica se a quantidade é 2 ou mais
verify_duplicates = (
    bronze_pedidos
    .groupBy("id_pedido")
    .count()
    .filter("count > 1")
)

verify_duplicates.display()

In [0]:
# Verifica os dados nulos
# O código faz uma contagem de quantos campos estão vazios
bronze_pedidos.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in bronze_pedidos.columns
]).show()

In [0]:
# Fazendo o join para verificar se o id_cliente não referencia nenhuma linha em bronze_clientes
bronze_clientes = spark.read.table('rocket.bronze.tb_clientes')

invalid_clients = (
    bronze_pedidos
    .join(
        bronze_clientes,
        bronze_pedidos.id_cliente == bronze_clientes.id_cliente,
        "left_anti"
    )
)

invalid_clients.display()

In [0]:
# Fazendo o join para verificar se o id_produto não referencia nenhuma linha em bronze_produtos
bronze_produtos = spark.read.table('rocket.bronze.tb_catalogo_produtos')

invalid_orders = (
    bronze_pedidos
    .join(
        bronze_produtos,
        bronze_pedidos.id_produto == bronze_produtos.id_produto,
        "left_anti"
    )
)

invalid_orders.display()

In [0]:
# Verifica quantas instâncias não são float
verify_formating_cost = (
    bronze_pedidos
    .select(
        sum(when(F.col("valor_pedido").try_cast("float").isNull(), 1).otherwise(0)).alias("invalid_float_count"),
        F.count("*").alias("total_count")
    )
)

verify_formating_cost.display()

In [0]:
from pyspark.sql.functions import regexp_replace, substring

# Aplica a substituição de ',' por '.' e remove os dois primeiros caracteres (R$)
bronze_pedidos_clean = bronze_pedidos.withColumn(
    "valor_pedido_clean",
    regexp_replace(regexp_replace(col("valor_pedido"), "^R\\$", ""), ",", ".")
)

# Filtra instâncias corretas: valor_pedido_clean pode ser convertido para float e segue o padrão numérico
wrong_instances = bronze_pedidos_clean.filter(
    col("valor_pedido_clean").try_cast("float").isNull() &
    col("valor_pedido_clean").rlike(r"^\d+(\.\d+)?$")
)

# Conta as instâncias corretas
wrong_instances.display()

In [0]:
# Verificando os valores negativos
# Possível tratamento é converter para positivo (abs)
bronze_pedidos_cost = (
    bronze_pedidos_clean.filter(col("valor_pedido").try_cast("float") < 0)
)

bronze_pedidos_cost.display()

In [0]:
# Verificando se valor pedido tem 0
bronze_pedidos_cost = (
    bronze_pedidos_clean.filter(col("valor_pedido").try_cast("float") == 0)
)

bronze_pedidos_cost.display()

In [0]:
from pyspark.sql.functions import col

# Verifica datas que não estão no formato YYYY-MM-DD
invalid_date_format = bronze_pedidos.filter(~col("data_pedido").rlike(r"^\d{4}-\d{2}-\d{2}$"))

count_invalid = invalid_date_format.count()

invalid_date_format.display()
display(count_invalid)

In [0]:
# Testa uma possível abordagem de correção dos dados, padronizando para YYYY/MM/DD
from pyspark.sql.functions import split, concat_ws, col, when, length, regexp_replace

# Troca '-' por '/' na data_pedido para padronizar
bronze_pedidos_data = bronze_pedidos.withColumn(
    "data_pedido",
    regexp_replace(col("data_pedido"), "/", "-")
)

# Divide a data_pedido em partes
parts = split(col("data_pedido"), "-")

# Identifica o ano (4 dígitos) e marca a posição
bronze_pedidos_padronizado = bronze_pedidos_data.withColumn(
    "ano",
    when(length(parts.getItem(0)) == 4, parts.getItem(0))
    .when(length(parts.getItem(1)) == 4, parts.getItem(1))
    .when(length(parts.getItem(2)) == 4, parts.getItem(2))
).withColumn(
    "pos_ano",
    when(length(parts.getItem(0)) == 4, 0)
    .when(length(parts.getItem(1)) == 4, 1)
    .when(length(parts.getItem(2)) == 4, 2)
)

# Identifica os dois valores que não são ano
bronze_pedidos_padronizado = bronze_pedidos_padronizado.withColumn(
    "valor1",
    when(col("pos_ano") == 0, parts.getItem(1))
    .when(col("pos_ano") == 1, parts.getItem(0))
    .otherwise(parts.getItem(0))
).withColumn(
    "valor2",
    when(col("pos_ano") == 0, parts.getItem(2))
    .when(col("pos_ano") == 1, parts.getItem(2))
    .otherwise(parts.getItem(1))
)

# Aplica a regra: se algum > 12, é dia; senão, valor1 = mês, valor2 = dia
bronze_pedidos_padronizado = bronze_pedidos_padronizado.withColumn(
    "dia",
    when(col("valor1").try_cast("int") > 12, col("valor1"))
    .when(col("valor2").try_cast("int") > 12, col("valor2"))
    .otherwise(col("valor2"))  # Se ambos <= 12, escolhe valor2 como dia
).withColumn(
    "mes",
    when(col("valor1").try_cast("int") > 12, col("valor2"))
    .when(col("valor2").try_cast("int") > 12, col("valor1"))
    .otherwise(col("valor1"))  # Se ambos <= 12, escolhe valor1 como mês
)

# Monta a data padronizada YYYY/MM/DD
bronze_pedidos_padronizado = bronze_pedidos_padronizado.withColumn(
    "data_pedido_padronizada",
    concat_ws("-", col("ano"), col("mes"), col("dia"))
)

# Verifica instâncias ainda incorretas (formato ou valores inválidos)
instancias_incorretas = bronze_pedidos_padronizado.filter(
    ~col("data_pedido_padronizada").rlike(r"^\d{4}-\d{2}-\d{2}$") |
    (col("mes").try_cast("int") < 1) | (col("mes").try_cast("int") > 12) |
    (col("dia").try_cast("int") < 1) | (col("dia").try_cast("int") > 31)
)

instancias_incorretas.select("ano", "mes", "dia", "data_pedido_padronizada", "data_pedido").display()
bronze_pedidos_padronizado.select(col("data_pedido_padronizada"), col("data_pedido")).display()

In [0]:
# Verificando se existem casos com data inválidas ou datas futuras
from pyspark.sql.functions import to_date, current_date, col, when

fixed_date = to_date("2026-04-28", "yyyy-MM-dd")

# Tenta converter para data válida
bronze_pedidos_padronizado = bronze_pedidos_padronizado.withColumn(
    "data_pedido_valid",
    to_date(col("data_pedido_padronizada"), "yyyy-MM-dd")
)

# Verifica datas inválidas (não existem no calendário)
invalid_date = bronze_pedidos_padronizado.filter(col("data_pedido_valid").isNull())

# Verifica datas futuras
future_date = bronze_pedidos_padronizado.filter(
    col("data_pedido_valid").isNotNull() & (col("data_pedido_valid") > fixed_date)
)

# Marca o tipo de erro
bronze_pedidos_padronizado = bronze_pedidos_padronizado.withColumn(
    "caso_invalido",
    when(col("data_pedido_valid").isNull(), "data_invalida")
    .otherwise(None)
).withColumn(
    "caso_futuro",
    when(col("data_pedido_valid").isNotNull() & (col("data_pedido_valid") > current_date()), "data_futura")
    .otherwise(None)
)

# Exibe as instâncias inválidas ou futuras
display(
    bronze_pedidos_padronizado.filter(
        col("caso_invalido").isNotNull() | col("caso_futuro").isNotNull()
    ).select(
        "data_pedido_padronizada", "data_pedido", "caso_invalido", "caso_futuro"
    )
)

As data futuras talvez não seja necessário tratar, pois todas estão no mês de maio, então é razoável manter

In [0]:
# Verificar as variações de método de pagamento
bronze_pedidos.select("metodo_pagamento").distinct().display()

# Sugestão - Colocar tudo para UPPER e um dicionário para outros casos especiais,
# por serem poucas instâncias

In [0]:
# Verificar as variações no status
bronze_pedidos.select("status").distinct().display()

# Mesmo tratamento do método de pagamento

In [0]:
from pyspark.sql.functions import col

# Filtra valores de 'quantidade' que não podem ser convertidos para int
invalid_quantidade = bronze_pedidos.filter(
    col("quantidade").try_cast("int").isNull()
).select("quantidade").distinct()

display(invalid_quantidade)

In [0]:
# Verificar se há valores negativos
negative_values = bronze_pedidos.filter(col("quantidade").try_cast("int") < 0).select("quantidade")

negative_values.display()
# Pode ser o mesmo tratamento que o de valores negativos, que é manter o absoluto

## Aplicando as transformações para a silver

In [0]:
from pyspark.sql.functions import (
    col, lit, when, coalesce,
    regexp_replace, expr,
    abs as spark_abs,
    round as spark_round,
    create_map,
    to_date, split, length, concat_ws,
    lower, trim
)

In [0]:
def clean_cost(cost_col):
    """Remove R$ e vírgula, converte para float, retorna valor absoluto e arredonda para duas casas decimais"""
    temp_col = regexp_replace(regexp_replace(cost_col, "^R\\$", ""), ",", ".")
    return spark_round(spark_abs(temp_col.cast("float")), 2)

In [0]:
def clean_date(date_col):
    """Padroniza datas para o formato YYYY-MM-DD e valida datas futuras"""
    fixed_date = to_date(lit("2026-04-28"), "yyyy-MM-dd")

    date_col = regexp_replace(date_col, "/", "-")
    parts = split(date_col, "-")

    ano = when(length(parts.getItem(0)) == 4, parts.getItem(0)) \
        .when(length(parts.getItem(1)) == 4, parts.getItem(1)) \
        .otherwise(parts.getItem(2))

    pos_ano = when(length(parts.getItem(0)) == 4, 0) \
        .when(length(parts.getItem(1)) == 4, 1) \
        .otherwise(2)

    valor1 = when(pos_ano == 0, parts.getItem(1)) \
        .when(pos_ano == 1, parts.getItem(0)) \
        .otherwise(parts.getItem(0))

    valor2 = when(pos_ano == 0, parts.getItem(2)) \
        .when(pos_ano == 1, parts.getItem(2)) \
        .otherwise(parts.getItem(1))

    # ✅ CORREÇÃO: .cast("int") no lugar de .try_cast() que não existe no PySpark
    dia = when(valor1.cast("int") > 12, valor1) \
        .when(valor2.cast("int") > 12, valor2) \
        .otherwise(valor2)

    mes = when(valor1.cast("int") > 12, valor2) \
        .when(valor2.cast("int") > 12, valor1) \
        .otherwise(valor1)

    data_padronizada = concat_ws("-", ano, mes, dia)
    data_valida = to_date(data_padronizada, "yyyy-MM-dd")

    # ✅ Adicionado isNull() para capturar datas mal formadas
    return when(
        data_valida > fixed_date,
        lit(None)
    ).otherwise(data_padronizada)

In [0]:
def clean_payment_method(payment_method_col):
    """Padroniza o método de pagamento"""
    map_dict = {
        "crt": "cartao",
        "b0leto": "boleto",
        "p1x": "pix",
        "c@rtao": "cartao",
        "cartão":"cartao",
        "bol": "boleto"
    }
    
    # Cria um mapa literal do Spark (mais eficiente que iterar)
    mapping_expr = create_map([lit(x) for pair in map_dict.items() for x in pair])
    
    # Normaliza para lowercase e remove espaços
    temp_col = trim(lower(payment_method_col))
    
    # Aplica o mapa, mantendo o valor original se não houver correspondência
    return coalesce(mapping_expr[temp_col], temp_col)

In [0]:
def clean_status(status_col):
    """Padroniza o status"""
    map_dict = {
        "aprovadoo": "aprovado",
        "aprov": "aprovado",
        "apr": "aprovado",
        "process": "processado",
        "proc": "processado",
        "recus": "recusado",
        "recusadoo": "recusado",
        "rec": "recusado",
        "reembolsad": "reembolsado",
        "reemb": "reembolsado",
        "reembolso": "reembolsado"
    }
    
    # Cria um mapa literal do Spark (mais eficiente que iterar)
    mapping_expr = create_map([lit(x) for pair in map_dict.items() for x in pair])
    
    # Normaliza para lowercase e remove espaços
    temp_col = trim(lower(status_col))
    
    # Aplica o mapa, mantendo o valor original se não houver correspondência
    return coalesce(mapping_expr[temp_col], temp_col)

In [0]:
def clean_quantity(df):
    bronze_produtos = spark.read.table('rocket.bronze.tb_catalogo_produtos')

    map_dict = {"cinco": "5", "um": "1", "quatro": "4"}
    mapping_expr = create_map([lit(x) for pair in map_dict.items() for x in pair])

    df_temp = df.withColumn(
        "quantidade_temp",
        coalesce(mapping_expr[col("quantidade")], col("quantidade"))
    )

    df_temp = df_temp.withColumn(
        "quantidade_float",
        expr("TRY_CAST(quantidade_temp AS FLOAT)")
    )

    # CORREÇÃO: limpa o preço antes do join (troca vírgula por ponto e converte)
    bronze_produtos_typed = bronze_produtos.select(
                col("id_produto").alias("id_produto_join"),
                        when(trim(lower(col("preco"))) == "null",lit(None).cast("float")).otherwise(
                        regexp_replace(regexp_replace(regexp_replace(col("preco"), r"^R\$\s*", ""), ",", "."), " ", "").cast("float")).alias("preco_produto")
    )

    df_temp = df_temp.withColumn("id_produto_join", col("id_produto"))

    df_with_price = df_temp.join(
        bronze_produtos_typed,
        "id_produto_join",
        "left"
    )

    df_with_price = df_with_price.withColumn(
        "valor_pedido_float",
        regexp_replace(
            regexp_replace(col("valor_pedido"), r"^R\$\s*", ""), ",", "."
        ).cast("float")
    )

    df_final = df_with_price.withColumn(
        "quantidade_limpa",
        when(
            col("quantidade_float").isNull() | (col("quantidade_float") == 0.0),
            when(
                col("preco_produto").isNull() | (col("preco_produto") == 0.0),
                lit(None).cast("long")
            ).otherwise(
                spark_round(
                    spark_abs(col("valor_pedido_float")) / spark_abs(col("preco_produto"))
                ).cast("long")
            )
        ).otherwise(
            spark_abs(col("quantidade_float")).cast("long")
        )
    )

    return (
        df_final
        .drop("quantidade_temp", "quantidade_float", "preco_produto",
              "valor_pedido_float", "id_produto_join")
        .withColumn("quantidade", col("quantidade_limpa"))
        .drop("quantidade_limpa")
    )

In [0]:
# Carrega a tabela bronze de pedidos
bronze_pedidos = spark.read.table("rocket.bronze.tb_pedidos")  # ajuste o nome se for diferente

# Limpa a quantidade
bronze_pedidos_clean_quantity = clean_quantity(bronze_pedidos)

silver_fat_pedidos = bronze_pedidos_clean_quantity.select(
    "id_pedido",
    "id_produto",
    "id_cliente",
    clean_cost(col("valor_pedido")).alias("valor_pedido"),
    clean_date(col("data_pedido")).alias("data_pedido"),
    clean_payment_method(col("metodo_pagamento")).alias("metodo_pagamento"),
    clean_status(col("status")).alias("status"),
    col("quantidade"),
    F.current_timestamp().alias("timestamp_ingestion")
)

silver_fat_pedidos.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("rocket.silver.fat_pedidos")

In [0]:
spark.read.table("rocket.silver.fat_pedidos").display()

In [0]:
import pyspark.sql.functions as F

silver_pedidos = spark.read.table("rocket.silver.fat_pedidos")
bronze_produtos = spark.read.table("rocket.bronze.tb_catalogo_produtos")

df_join = silver_pedidos.join(
    bronze_produtos.select("id_produto", "preco"),
    "id_produto",
    "inner"
).filter(col("quantidade").isNull()) \
 .select("quantidade", "preco")

# Converte preco para double, tratando "null" como None
df_join = df_join.withColumn(
    "preco_num",
    when(lower(trim(col("preco"))) == "null", lit(None))
    .otherwise(col("preco").cast("double"))
)

df_join = df_join.select(
    F.count('*').alias("quantidade_null"),
    F.sum(when((col("preco_num").isNull()) | (col("preco_num") == 0), 1).otherwise(0)).alias("total_preco_null")
)

display(df_join)

# Tabela Suporte

In [0]:
from pyspark.sql import functions as F

# leitura da tabela na camada Bronze
df_bronze_tickets = spark.table("rocket.bronze.tb_suporte_tickets")

# Função auxiliar para remover acentos mapeando os caracteres
def remover_acentos(coluna):
    acentos = "áàâãäéèêëíìîïóòôõöúùûüçÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇ"
    sem_acentos = "aaaaaeeeeiiiiooooouuuucAAAAAEEEEIIIIOOOOOUUUUC"
    return F.translate(coluna, acentos, sem_acentos)

df_silver_tickets = (
    df_bronze_tickets

    # RECALCULANDO O TEMPO DE RESOLUÇÃO (Corrigindo o bug da origem)
    # isso sobrescreve a coluna antiga com a matemática exata em horas
    # a coluna tempo_resolucao_horas estava inconsistente com o periodo da data_abertura com data_resolucao

    .withColumn("tempo_resolucao_horas",
        F.when(F.col("data_resolucao").isNotNull(),
            F.round(
                (F.unix_timestamp(F.to_timestamp("data_resolucao")) - 
                 F.unix_timestamp(F.to_timestamp("data_abertura"))) / 3600, 
            1) # Arredonda para 1 casa decimal (ex: 42.5 horas)
        ).otherwise(F.lit(None))
    )

    # STATUS
    .withColumn("status_ticket", 
        F.when(F.col("data_resolucao").isNull(), "aberto")
         .otherwise("fechado")
    )
    
    # DATAS
    .withColumn("data_abertura", F.date_format(F.to_date("data_abertura"), "yyyy-MM-dd"))
    .withColumn("data_resolucao", F.date_format(F.to_date("data_resolucao"), "yyyy-MM-dd"))
    
    # Garantindo que os nomes fiquem com a primeira letra maiúscula
    .withColumn("agente_suporte", F.initcap(F.trim(F.lower(F.col("agente_suporte")))))
    
    # Tratando as variações dos tipos_problema
    .withColumn("tipo_problema", F.trim(F.lower(F.col("tipo_problema"))))
    .withColumn("tipo_problema", remover_acentos(F.col("tipo_problema")))
    .withColumn("tipo_problema",
        F.when(F.col("tipo_problema").isin("produto", "product", "pro", "prod", "p3oduto"), "produto")
         .when(F.col("tipo_problema").isin("pagamento", "p4gamento", "pag", "pay", "payment"), "pagamento")
         .when(F.col("tipo_problema").isin("entrega", "del", "delay", "ent", "3ntrega", "entr"), "entrega")
         .when(F.col("tipo_problema").isin("reembolso", "reemb", "r3embolso", "ref", "refund"), "reembolso")
         .otherwise(F.col("tipo_problema"))
    )
    
    
)


In [0]:
display(df_silver_tickets)

In [0]:
# Gera estatísticas básicas: média, mínimo, máximo e desvio padrão
display(df_silver_tickets.select("tempo_resolucao_horas").summary())

In [0]:
# Filtro para encontrar registros com datas ilógicas ou obrigatórias vazias
df_datas_invalidas = df_silver_tickets.filter(
    (F.col("data_abertura").isNull()) | 
    (F.col("data_resolucao") < F.col("data_abertura"))
)

# contagem para ver o volume do problema
print(f"Total de registros com datas inválidas: {df_datas_invalidas.count()}")

# exibe as linhas problematicas, se houver
display(df_datas_invalidas)

In [0]:
# Filtra tickets que ESTÃO resolvidos, mas NÃO possuem o tempo calculado
df_inconsistencia_tempo = df_silver_tickets.filter(
    F.col("data_resolucao").isNotNull() & 
    F.col("tempo_resolucao_horas").isNull()
)

print(f"Tickets fechados sem tempo de resolução calculado: {df_inconsistencia_tempo.count()}")
display(df_inconsistencia_tempo)

In [0]:
display(df_silver_tickets.select("tipo_problema").distinct())

In [0]:
df_silver_tickets.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("rocket.silver.fat_suporte_tickets")

In [0]:
df_validacao = spark.table("rocket.silver.fat_suporte_tickets")
display(df_validacao.limit(10))

# Tabela Produtos

In [0]:
df_pedidos = spark.read.table('rocket.bronze.tb_pedidos')
df_pedidos.dtypes

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, when, trim, lower, lit, regexp_replace, abs as spark_abs, round as spark_round, coalesce, create_map
)

df_catalogo = spark.read.table('rocket.bronze.tb_catalogo_produtos')
df_pedidos = spark.read.table('rocket.bronze.tb_pedidos')

# funcao auxiliar para limpar preco (remove R$, troca virgula por ponto)
def limpar_preco(coluna):
    return regexp_replace(regexp_replace(regexp_replace(coluna, r"^R\$\s*", ""), ",", "."), " ", "")

# mapa de categorias validas
mapa_categorias = {
    "eletronicos": "Eletronicos", "eletronico": "Eletronicos", "eletrônico": "Eletronicos",
    "elet": "Eletronicos",
    "moveis": "Moveis", "movel": "Moveis", "mov": "Moveis", "m0veis": "Moveis",
    "brinquedos": "Brinquedos", "brinquedo": "Brinquedos", "brinq": "Brinquedos",
    "brin": "Brinquedos",
    "beleza": "Beleza", "bel": "Beleza", "belz": "Beleza", "b3leza": "Beleza",
    "esportes": "Esportes", "esporte": "Esportes", "esport": "Esportes",
    "esp": "Esportes", "3sportes": "Esportes",
    "automotivo": "Automotivo", "automotiv3": "Automotivo", "autom": "Automotivo",
    "aut": "Automotivo",
    "casa": "Casa", "cas": "Casa", "cas@": "Casa", "cas4": "Casa",
    "vestuario": "Vestuario", "vestuarios": "Vestuario", "vest": "Vestuario",
    "vestu": "Vestuario"
}

mapping_expr = create_map([lit(x) for pair in mapa_categorias.items() for x in pair])

# mapa para converter palavras em numeros na coluna quantidade
mapa_quantidade = create_map([
    lit(x) for pair in {
        "cinco": "5", "um": "1", "quatro": "4", "dois": "2", "tres": "3"
    }.items() for x in pair
])

# prepara pedidos para cruzamento (com tratamento de quantidade textual)
df_pedidos_prep = df_pedidos.select(
    col("id_produto").alias("id_produto_join"),
    limpar_preco(col("valor_pedido")).cast("double").alias("valor_pedido_num"),
    spark_abs(
        coalesce(mapa_quantidade[lower(trim(col("quantidade")))], col("quantidade"))
        .cast("double")
    ).alias("quantidade_num")
).filter(
    col("valor_pedido_num").isNotNull() & col("quantidade_num").isNotNull() &
    (col("valor_pedido_num") > 0) & (col("quantidade_num") > 0)
)

# agrega por produto: preco medio calculado via valor/quantidade
df_preco_referencia = df_pedidos_prep.withColumn(
    "preco_unitario", col("valor_pedido_num") / col("quantidade_num")
).groupBy("id_produto_join").agg(
    F.avg("preco_unitario").alias("preco_referencia")
)

# join do catalogo com preco de referencia dos pedidos
df_silver = df_catalogo.join(
    df_preco_referencia,
    df_catalogo.id_produto == df_preco_referencia.id_produto_join,
    "left"
).drop("id_produto_join")

# ativo: padroniza para booleano
df_silver = df_silver.withColumn(
    "ativo",
    when(lower(trim(col("ativo"))).isin("sim", "s", "1", "true"), True)
    .when(lower(trim(col("ativo"))).isin("nao", "n", "0", "false"), False)
    .otherwise(None)
)

# preco: nulo falso, abs nos negativos, zero vira preco de referencia dos pedidos
df_silver = df_silver.withColumn(
    "preco_num",
    when(lower(trim(col("preco"))) == "null", lit(None))
    .otherwise(limpar_preco(col("preco")).cast("double"))
).withColumn(
    "preco",
    when(col("preco_num").isNull(), lit(None))
    .when(col("preco_num") < 0, spark_abs(col("preco_num")))
    .when(col("preco_num") == 0,
        when(col("preco_referencia").isNotNull(), col("preco_referencia"))
        .otherwise(lit(None))
    )
    .otherwise(col("preco_num"))
    .cast("double")
).withColumn("preco", F.round(col("preco"), 2)
).drop("preco_num", "preco_referencia")


keywords_categoria = {
    "Eletronicos": (
        r"smartphone|notebook|tablet(?!.infantil)|fone|headset|câmera|camera|computador|monitor|"
        r"teclado|mouse|carregador|cabo\s*usb|roteador|caixa.de.som|console|videogame|impressora|"
        r"webcam|projetor|drone|smart.?tv|suporte.para.notebook|hub\s*usb|adaptador|bateria.port|"
        r"pen.drive|ssd|hd.externo|controle.sem.fio|placa.de.captura|leitor|switch.hdmi|microfone|"
        r"estabilizador|nobreak|smartband|smart.?watch|aspirador.rob"
    ),
    "Vestuario": (
        r"camiseta|camisa|calça|calca|vestido|blusa|jaqueta|casaco|short|bermuda|saia|meia|"
        r"cueca|sutiã|sutia|pijama|moletom|roupão|roupao|tênis|tenis|sapato|sandália|sandalia|"
        r"bota|chinelo|sapatilha|conjunto.esportivo|fashion.import|confecç|moda.brasil|textilbrasil"
    ),
    "Casa": (
        r"panela|frigideira|cobertor|travesseiro|lençol|lencol|toalha|vassoura|rodo|"
        r"liquidificador|batedeira|micro.?ondas|cafeteira|torradeira|geladeira|fogão|fogao|forno|"
        r"ventilador|ar.condicionado|cortina|luminária|luminaria|lâmpada|lampada|tábua|tabua|"
        r"multiprocessador|conjunto.de.panelas|kit.churrasco|difusor|aspirador.de.pó|aspirador.de.po|"
        r"utilidades.do.lar|homebrasil|casa.&.cia"
    ),
    "Esportes": (
        r"bola|chuteira|raquete|luva.esport|capacete|haltere|anilha|esteira|colchonete|"
        r"mochila.esportiva|garrafa.esportiva|joelheira|cotoveleira|patins|yoga|bloco.de.yoga|"
        r"sportsbrasil|atletica.import|fitness.distribuidora"
    ),
    "Beleza": (
        r"modelador.de.cachos|óleo.corporal|oleo.corporal|sabonete|creme.para|"
        r"shampoo(?!\s*automotivo)|condicionador|hidratante|perfume|batom|maquiagem|esmalte|"
        r"protetor.solar|sérum|serum|loção|locao|desodorante|beauty.import|cosméticos|cosmeticos|"
        r"perfumes.&.cia|beleza.total"
    ),
    "Automotivo": (
        r"oleo.de.motor|óleo.de.motor|pastilha.de.freio|shampoo.automotivo|pneu|filtro.de.ar|"
        r"farol|retrovisor|volante|tapete.automotivo|capa.de.banco|carregador.veicular|alarme.auto|"
        r"estepe|macaco.hidra|extintor|autopecas|peças.&.cia.distribuidora|automotivo.import"
    ),
    "Moveis": (
        r"mesa.de.centro|mesa.de.canto|sofá|sofa|cadeira|cama|armário|armario|estante|prateleira|"
        r"guarda.roupa|colchão|colchao|rack|escrivaninha|poltrona|cristaleira|cômoda|comoda|"
        r"beliche|berco|moveisplus|decor.brasil.atacado|interiores.import"
    ),
    "Brinquedos": (
        r"brinquedo|boneca|carrinho.de.brinc|lego|quebra.cabeça|massinha|pelúcia|pelucia|"
        r"fantoche|velocipede|tablet.infantil|fun.kids|brinquedos.do.brasil|toy.import"
    ),
}

def build_categoria_inference(df, col_nome, col_fornecedor):
    expr = lit(None).cast("string")
    for categoria, pattern in keywords_categoria.items():
        texto = F.concat_ws(" ", lower(trim(col(col_nome))), lower(trim(col(col_fornecedor))))
        expr = when(texto.rlike(f"(?i){pattern}"), lit(categoria)).otherwise(expr)
    return expr

df_silver = df_silver.withColumn(
    "categoria_norm", lower(trim(col("categoria")))
).withColumn(
    "categoria",
    when(col("categoria_norm") == "null", lit(None))
    .otherwise(coalesce(mapping_expr[col("categoria_norm")], lit(None)))
).drop("categoria_norm")

# Inferência: preenche apenas os nulls restantes
df_silver = df_silver.withColumn(
    "categoria_inferida",
    build_categoria_inference(df_silver, "nome_produto", "fornecedor")
).withColumn(
    "categoria",
    coalesce(col("categoria"), col("categoria_inferida"))
).drop("categoria_inferida")

# estoque_disponivel: nulo falso, converte para integer
df_silver = df_silver.withColumn(
    "estoque_num",
    when(lower(trim(col("estoque_disponivel"))) == "null", lit(None))
    .otherwise(col("estoque_disponivel").cast("integer"))
).withColumn(
    "estoque_disponivel", col("estoque_num").cast("integer")
).drop("estoque_num")

# ativo com estoque zero: marca como inativo
df_silver = df_silver.withColumn(
    "ativo",
    when((col("ativo") == True) & (col("estoque_disponivel") == 0), False)
    .otherwise(col("ativo"))
)

# peso_kg: nulo falso para nulo real, converte para double
df_silver = df_silver.withColumn(
    "peso_kg",
    when(lower(trim(col("peso_kg"))) == "null", lit(None))
    .otherwise(col("peso_kg").cast("double"))
)

# avaliacao_interna: nulo falso para nulo real, converte para double
df_silver = df_silver.withColumn(
    "avaliacao_interna",
    when(lower(trim(col("avaliacao_interna"))) == "null", lit(None))
    .otherwise(col("avaliacao_interna").cast("double"))
)

display(df_silver)

In [0]:
# Salvando a tabela na camada Silver
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("rocket.silver.dim_catalogo_produtos")

print("rocket.silver.dim_catalogo_produtos atualizada com sucesso!")

In [0]:
# validando a tabela salva na camada Silver
df_validacao = spark.table("rocket.silver.dim_catalogo_produtos")
display(df_validacao.limit(10))

# Tabela clickstream

In [0]:
#mapeamento e padronização
df = spark.table("rocket.bronze.tb_clickstream")
 
print(f"Total de linhas: {df.count():,}")
 
mapeamento = {
    # page_view
    "page_view": "page_view", "PAGE_VIEW": "page_view", "PageView": "page_view",
    "page-view": "page_view", "pageview":  "page_view", "pv":       "page_view",
    # search
    "search": "search", "SEARCH": "search", "Search": "search",
    "busca":  "search", "BUSCA":  "search", "srch":   "search",
    # product_view
    "product_view": "product_view", "PRODUCT_VIEW": "product_view", "ProductView": "product_view",
    "product-view": "product_view", "prod_view":    "product_view", "productview": "product_view",
    # add_to_cart
    "add_to_cart": "add_to_cart", "ADD_TO_CART": "add_to_cart", "AddToCart":  "add_to_cart",
    "add-to-cart": "add_to_cart", "adicionar":   "add_to_cart", "ADICIONAR":  "add_to_cart",
    # abandon_cart
    "abandon_cart": "abandon_cart", "ABANDON_CART": "abandon_cart", "AbandonCart": "abandon_cart",
    "abandon-cart": "abandon_cart", "abandono":     "abandon_cart", "ABANDONO":    "abandon_cart",
    # checkout
    "checkout":  "checkout", "CHECKOUT":  "checkout", "Checkout":  "checkout",
    "check_out": "checkout", "pagamento": "checkout", "PAGAMENTO": "checkout",
    # purchase
    "purchase": "purchase", "PURCHASE": "purchase", "Purchase": "purchase",
    "compra":   "purchase", "COMPRA":   "purchase", "buy":      "purchase",
    # login
    "login":  "login", "LOGIN":  "login", "Login":  "login",
    "log_in": "login", "signin": "login", "SIGNIN": "login",
}
 
mapping_expr = F.create_map(
    *[item for pair in mapeamento.items() for item in (F.lit(pair[0]), F.lit(pair[1]))]
)
 
df_silver = df.withColumn(
    "tipo_evento",
    F.coalesce(mapping_expr[F.col("tipo_evento")], F.col("tipo_evento"))
)
 
display(df_silver)

In [0]:
#padronizacao das palavras das duas colunas e ajuste de combinações impossíveis
from pyspark.sql import functions as F
 
df = df_silver # spark.table("rocket.bronze.tb_clickstream")
 
map_canal = F.create_map(
    F.lit("web"),        F.lit("web"),
    F.lit("Web"),        F.lit("web"),
    F.lit("WEB"),        F.lit("web"),
    F.lit("app"),        F.lit("app"),
    F.lit("App"),        F.lit("app"),
    F.lit("APP"),        F.lit("app"),
    F.lit("aplicativo"), F.lit("app"),
    F.lit("mobile_web"), F.lit("mobile_web"),
    F.lit("Mobile_Web"), F.lit("mobile_web"),
    F.lit("MOBILE_WEB"), F.lit("mobile_web"),
    F.lit("mobile web"), F.lit("mobile_web"),
)
 
map_dispositivo = F.create_map(
    F.lit("desktop"),    F.lit("desktop"),
    F.lit("Desktop"),    F.lit("desktop"),
    F.lit("DESKTOP"),    F.lit("desktop"),
    F.lit("computador"), F.lit("desktop"),
    F.lit("mobile"),     F.lit("mobile"),
    F.lit("MOBILE"),     F.lit("mobile"),
    F.lit("celular"),    F.lit("mobile"),
    F.lit("mob"),        F.lit("mobile"),
    F.lit("tablet"),     F.lit("tablet"),
    F.lit("Tablet"),     F.lit("tablet"),
    F.lit("TABLET"),     F.lit("tablet"),
    F.lit("tab"),        F.lit("tablet"),
)
 
df_norm = df \
    .withColumn("canal",       F.coalesce(map_canal[F.col("canal")],             F.col("canal"))) \
    .withColumn("dispositivo", F.coalesce(map_dispositivo[F.col("dispositivo")], F.col("dispositivo")))
 
df_silver = df_norm \
    .withColumn(
        "dispositivo",
        F.when(
            (F.col("canal") == "app") & (F.col("dispositivo") == "desktop"),
            F.lit("mobile")                  # app não roda em desktop → mobile
        ).otherwise(F.col("dispositivo"))
    ) \
    .withColumn(
        "canal",
        F.when(
            (F.col("canal") == "web") & (F.col("dispositivo") == "mobile"),
            F.lit("mobile_web")              # web + mobile → mobile_web
        ).when(
            (F.col("canal") == "mobile_web") & (F.col("dispositivo") == "tablet"),
            F.lit("web")                     # mobile_web + tablet → web
        ).otherwise(F.col("canal"))
    )
 
display(df_silver)

In [0]:
#muda nulos, zeros e negativos para a mediana
df = df_silver # spark.table("rocket.bronze.tb_clickstream")
 
mediana = df.filter(
    F.col("tempo_pagina_seg").isNotNull() &
    (F.col("tempo_pagina_seg") > 0)
).approxQuantile("tempo_pagina_seg", [0.5], 0.001)[0]
 
mediana = int(mediana)
 
df_silver = df.withColumn(
    "tempo_pagina_seg",
    F.when(
        F.col("tempo_pagina_seg").isNull() |
        (F.col("tempo_pagina_seg") <= 0),
        F.lit(mediana)
    ).otherwise(F.col("tempo_pagina_seg"))
)
 
display(df_silver)

In [0]:
df = df_silver # spark.table("rocket.bronze.tb_clickstream")

eventos_compra = ["checkout", "purchase", "compra", "pagamento", "buy",
                  "CHECKOUT", "PURCHASE", "COMPRA", "PAGAMENTO", "Checkout",
                  "Purchase", "check_out"]

df_silver = df \
    .withColumn(
        "flag_anonimo",
        F.col("id_cliente").isNull() & ~F.col("tipo_evento").isin(eventos_compra)
    ) \
    .withColumn(
        "flag_sem_produto",
        F.col("id_produto").isNull() & ~F.col("tipo_evento").isin(eventos_compra)
    ) \
    .filter(
        ~(F.col("id_cliente").isNull() & F.col("tipo_evento").isin(eventos_compra)) &
        ~(F.col("id_produto").isNull() & F.col("tipo_evento").isin(eventos_compra))
    )

print("=== Contagem das flags ===")
df_silver.select(
    F.count(F.when(F.col("flag_anonimo"),     True)).alias("flag_anonimo"),
    F.count(F.when(F.col("flag_sem_produto"), True)).alias("flag_sem_produto"),
).show()

print("=== Linhas removidas (críticos) ===")
print(f"flag_anonimo_critico + flag_sem_produto_critico removidos: {df.count() - df_silver.count():,} linhas")

display(df_silver)

In [0]:
(
    df_silver
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("rocket.silver.fat_clickstream")
)
 
print("rocket.silver.fat_clickstream atualizada com sucesso!") 